In [1]:
# Write PySpark code that:
#1. Reads the user_activity.csv file into a DataFrame.
#2. Converts the "activity_date" column to a date type.
#3. Determines the signup cohort for each user (the month of their signup).
#4. Calculates the number of users in each signup cohort who made a purchase in each subsequent month.
#5. Shows the results, including the signup cohort, activity month, and number of users.

In [2]:
from pyspark.sql import SparkSession

In [68]:
from pyspark.sql.functions import col, to_date, when, month, date_format, last, date_trunc, countDistinct
from pyspark.sql.window import Window

In [3]:
spark = SparkSession.builder.appName("Cohort").getOrCreate()

25/03/31 23:05:39 WARN Utils: Your hostname, Nikhils-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.0.90 instead (on interface en0)
25/03/31 23:05:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/31 23:05:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/03/31 23:05:52 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [19]:
df = spark.read.csv("user_activity.csv", header=True)

In [9]:
df.show()

+-------+-------------+-------------+
|user_id|activity_date|activity_type|
+-------+-------------+-------------+
|      1|   2023-01-05|       signup|
|      1|   2023-01-10|     purchase|
|      1|   2023-02-15|     purchase|
|      2|   2023-01-10|       signup|
|      2|   2023-01-20|     purchase|
|      3|   2023-01-15|       signup|
|      3|   2023-02-01|     purchase|
|      4|   2023-02-01|       signup|
|      4|   2023-02-10|     purchase|
|      5|   2023-02-15|       signup|
|      5|   2023-03-01|     purchase|
+-------+-------------+-------------+



In [22]:
df.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- activity_date: date (nullable = true)
 |-- activity_type: string (nullable = true)



In [40]:
df = df.withColumn("activity_date", to_date(col("activity_date")))

In [21]:
df = df.withColumn("activity_date", col("activity_date").cast("Date"))

In [41]:
window_spec = Window.partitionBy("user_id").orderBy("activity_date")

In [29]:
#df = df.withColumn("cohort_month", when(col("activity_type")=="signup", month("activity_date")))

In [51]:
cohort_df = df.withColumn("cohort_month", last(when(col("activity_type")=="signup", 
                                               date_trunc("month",col("activity_date"))), ignorenulls=True).over(window_spec))

In [52]:
cohort_df.show()

+-------+-------------+-------------+-------------------+
|user_id|activity_date|activity_type|       cohort_month|
+-------+-------------+-------------+-------------------+
|      1|   2023-01-05|       signup|2023-01-01 00:00:00|
|      1|   2023-01-10|     purchase|2023-01-01 00:00:00|
|      1|   2023-02-15|     purchase|2023-01-01 00:00:00|
|      2|   2023-01-10|       signup|2023-01-01 00:00:00|
|      2|   2023-01-20|     purchase|2023-01-01 00:00:00|
|      3|   2023-01-15|       signup|2023-01-01 00:00:00|
|      3|   2023-02-01|     purchase|2023-01-01 00:00:00|
|      4|   2023-02-01|       signup|2023-02-01 00:00:00|
|      4|   2023-02-10|     purchase|2023-02-01 00:00:00|
|      5|   2023-02-15|       signup|2023-02-01 00:00:00|
|      5|   2023-03-01|     purchase|2023-02-01 00:00:00|
+-------+-------------+-------------+-------------------+



In [53]:
purchase_df = cohort_df.filter(col("activity_type")=="purchase")

In [54]:
purchase_df.show()

+-------+-------------+-------------+-------------------+
|user_id|activity_date|activity_type|       cohort_month|
+-------+-------------+-------------+-------------------+
|      1|   2023-01-10|     purchase|2023-01-01 00:00:00|
|      1|   2023-02-15|     purchase|2023-01-01 00:00:00|
|      2|   2023-01-20|     purchase|2023-01-01 00:00:00|
|      3|   2023-02-01|     purchase|2023-01-01 00:00:00|
|      4|   2023-02-10|     purchase|2023-02-01 00:00:00|
|      5|   2023-03-01|     purchase|2023-02-01 00:00:00|
+-------+-------------+-------------+-------------------+



In [55]:
purchase_activity = purchase_df.withColumn("activity_month", date_trunc("month", col("activity_date")))

In [56]:
purchase_activity.show()

+-------+-------------+-------------+-------------------+-------------------+
|user_id|activity_date|activity_type|       cohort_month|     activity_month|
+-------+-------------+-------------+-------------------+-------------------+
|      1|   2023-01-10|     purchase|2023-01-01 00:00:00|2023-01-01 00:00:00|
|      1|   2023-02-15|     purchase|2023-01-01 00:00:00|2023-02-01 00:00:00|
|      2|   2023-01-20|     purchase|2023-01-01 00:00:00|2023-01-01 00:00:00|
|      3|   2023-02-01|     purchase|2023-01-01 00:00:00|2023-02-01 00:00:00|
|      4|   2023-02-10|     purchase|2023-02-01 00:00:00|2023-02-01 00:00:00|
|      5|   2023-03-01|     purchase|2023-02-01 00:00:00|2023-03-01 00:00:00|
+-------+-------------+-------------+-------------------+-------------------+



In [73]:
filtered_purchases = purchase_activity.filter(col("activity_month") > col("cohort_month"))

In [74]:
filtered_purchases.show()

+-------+-------------+-------------+-------------------+-------------------+
|user_id|activity_date|activity_type|       cohort_month|     activity_month|
+-------+-------------+-------------+-------------------+-------------------+
|      1|   2023-02-15|     purchase|2023-01-01 00:00:00|2023-02-01 00:00:00|
|      3|   2023-02-01|     purchase|2023-01-01 00:00:00|2023-02-01 00:00:00|
|      5|   2023-03-01|     purchase|2023-02-01 00:00:00|2023-03-01 00:00:00|
+-------+-------------+-------------+-------------------+-------------------+



In [75]:
result_df = filtered_purchases.groupBy("cohort_month", "activity_month").agg(countDistinct("user_id").alias("user_count")).orderBy("cohort_month", "activity_month")

In [76]:
result_df.show()

+-------------------+-------------------+----------+
|       cohort_month|     activity_month|user_count|
+-------------------+-------------------+----------+
|2023-01-01 00:00:00|2023-02-01 00:00:00|         2|
|2023-02-01 00:00:00|2023-03-01 00:00:00|         1|
+-------------------+-------------------+----------+

